
This notebook adds a small LLM-based comparison to the project. The main numerical fairness analysis is done using SBERT because SBERT gives stable cosine similarity scores that are easy to compare across original and counterfactual resumes.

However, since the project feedback suggested examining a current LLM, this notebook uses a lightweight open-source text-to-text model to generate a qualitative resume-job match explanation. The purpose is not to replace the SBERT scoring pipeline, but to show how an LLM can provide human-readable reasoning about whether a resume fits a job description.

This LLM demo is kept separate from the fairness metric calculation because generative LLM outputs can vary and may not always produce consistent numerical scores.

In [4]:
# Install the required libraries for the LLM demo
!pip install transformers torch pandas

In [5]:
import os
import pandas as pd
from transformers import pipeline
os.makedirs("data", exist_ok=True)

In [6]:
# Uploading the job and resume files
from google.colab import files
uploaded = files.upload()
for filename in uploaded.keys():
    os.rename(filename, f"data/{filename}")
print("Files uploaded and moved to the data folder.")

Saving jobs.csv to jobs.csv
Saving resume_variants.csv to resume_variants.csv
Files uploaded and moved to the data folder.


In [7]:
# Load the job descriptions and resume variants
jobs = pd.read_csv("data/jobs.csv")
resumes = pd.read_csv("data/resume_variants.csv")
print("Jobs shape:", jobs.shape)
print("Resume variants shape:", resumes.shape)
display(jobs.head())
display(resumes.head())

Jobs shape: (5, 5)
Resume variants shape: (40, 8)


,job_id,domain,title,company_name,job_description
0,J1,Software Engineering,Software Engineer,GOYT,Job Description:GOYT is seeking a skilled and ...
1,J2,Finance,Financial Analyst,Aya Healthcare,Aya Healthcare has an exciting 26-week contrac...
2,J3,Marketing,Marketing Coordinator,Corcoran Sawyer Smith,Job descriptionA leading real estate firm in N...
3,J4,Healthcare,Clinical Data Analyst,Insight Global,Must Have: 2+ years of current experience in a...
4,J5,Education,Academic Advisor,Kennesaw State University,About Us Are you ready to join a community lea...


,resume_id,domain,version,changed_signal,name,pronouns,university,resume_text
0,R1,Software Engineering,original,none,Emily Johnson,she/her,Stanford University,"Emily Johnson, she/her. B.S. in Computer Scien..."
1,R1,Software Engineering,name_changed,name,Aisha Khan,she/her,Stanford University,"Aisha Khan, she/her. B.S. in Computer Science ..."
2,R1,Software Engineering,pronoun_changed,pronoun,Emily Johnson,he/him,Stanford University,"Emily Johnson, he/him. B.S. in Computer Scienc..."
3,R1,Software Engineering,university_changed,university,Emily Johnson,she/her,Morgan State University,"Emily Johnson, she/her. B.S. in Computer Scien..."
4,R2,Software Engineering,original,none,Michael Smith,he/him,Massachusetts Institute of Technology,"Michael Smith, he/him. B.S. in Software Engine..."


In [1]:
# Load an open-source instruction-tuned language model for the LLM demo
# This model is used to generate a qualitative explanation of resume-job fit without requiring an external API key
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print("Open-source instruction-tuned LLM loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Open-source instruction-tuned LLM loaded successfully.


In [8]:
# Run the LLM on multiple original resume-job pairs as a qualitative comparison
llm_results = []
# Use only original resumes for the LLM comparison
original_resumes = resumes[resumes["version"] == "original"].copy()
for _, resume_row in original_resumes.iterrows():
    # Match each resume with a job from the same domain
    matching_jobs = jobs[jobs["domain"] == resume_row["domain"]]
    if len(matching_jobs) == 0:
        continue
    job_row = matching_jobs.iloc[0]
    resume_text_sample = str(resume_row["resume_text"])[:700]
    job_text_sample = (
        str(job_row["title"]) + " " +
        str(job_row["domain"]) + " " +
        str(job_row["company_name"]) + " " +
        str(job_row["job_description"]))[:700]
    prompt = f"""Decide whether the resume is a Strong match, Partial match, or Weak match for the job.
Job description:
{job_text_sample}
Resume:
{resume_text_sample}
Answer using only one phrase:
Strong match
Partial match
Weak match
"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=20,
        num_beams=4,
        early_stopping=True
    )
    decision = tokenizer.decode(outputs[0], skip_special_tokens=True)

    llm_results.append({
        "resume_id": resume_row["resume_id"],
        "resume_domain": resume_row["domain"],
        "job_id": job_row["job_id"],
        "job_title": job_row["title"],
        "job_domain": job_row["domain"],
        "llm_match_decision": decision
    })
llm_results_df = pd.DataFrame(llm_results)
display(llm_results_df)
print("Total LLM examples evaluated:", len(llm_results_df))

,resume_id,resume_domain,job_id,job_title,job_domain,llm_match_decision
0,R1,Software Engineering,J1,Software Engineer,Software Engineering,Weak match
1,R2,Software Engineering,J1,Software Engineer,Software Engineering,Weak match
2,R3,Finance,J2,Financial Analyst,Finance,Weak match
3,R4,Finance,J2,Financial Analyst,Finance,Weak match
4,R5,Marketing,J3,Marketing Coordinator,Marketing,Weak match
5,R6,Marketing,J3,Marketing Coordinator,Marketing,Weak match
6,R7,Healthcare,J4,Clinical Data Analyst,Healthcare,Weak match
7,R8,Healthcare,J4,Clinical Data Analyst,Healthcare,Weak match
8,R9,Education,J5,Academic Advisor,Education,Weak match
9,R10,Education,J5,Academic Advisor,Education,Weak match


Total LLM examples evaluated: 10


In [9]:
# Save the LLM match decisions
os.makedirs("results", exist_ok=True)
llm_results_df.to_csv("results/llm_match_decisions.csv", index=False)
print("LLM match decisions saved to results/llm_match_decisions.csv")

LLM match decisions saved to results/llm_match_decisions.csv


In [10]:
from google.colab import files
files.download("results/llm_match_decisions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>